# 94. 梯度提升模型

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 9 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 随机森林  →  **本章任务：** 梯度提升模型  →  **下一步：** 支持向量机（SVM）
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

前面章节我们已会用“多棵树投票”的随机森林做预测，但它更擅长把整体规律学得“平”一些。



## 本章目标

学完本章，你将能够：

- **理解**：理解「梯度提升模型」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「梯度提升模型」的关键输出指标。
- **迁移**：能把「梯度提升模型」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 94.1 核心概念

**背景引入**：前面章节我们已会用“多棵树投票”的随机森林做预测，但它更擅长把整体规律学得“平”一些。梯度提升是另一条思路：一棵树学完，把没猜对的样本交给下一棵树去补救，几棵树接力下来，往往能逼近更强的预测精度。这一章就在乳腺癌数据上用 HistGradientBoosting 动手体会它的威力，并学会用学习率和迭代轮数控制它的复杂度。

- 每一轮新模型纠正当前模型的错误（打个比方：像接力跑，上一棒没跑好、漏掉的地方，交给下一棒专门去补；几棒接力下来，整体就跑得更稳更快。）
- 较小学习率通常需要更多迭代
- 树深和叶节点数控制交互复杂度
- 提升模型对参数较敏感，应通过交叉验证选择


## 94.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 梯度提升分类 | `boost.predict_proba()`、`.fit()` | HistGradientBoosting 对中大型表格数据采用直方图加速。 | 在小测试集上细调大量参数 |
| 学习率与迭代数 | `rows.append()`、`m.predict_proba()`、`pd.DataFrame()`、`.fit()` | 比较有限配置，正式调参应放进交叉验证。 | 使用过深基学习器导致过拟合 |


## 94.3 示例 1：梯度提升分类

HistGradientBoosting 对中大型表格数据采用直方图加速。


<!-- math-foundation:chapter-94 -->
### 数学推导｜提升模型逐步拟合残差

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜从简单初始模型开始。** 例如平方损失下 $F_0(x)$ 可取目标均值。

**第 2 步｜计算当前模型还没解释的方向。** 一般损失下使用负梯度

$$
r_{im}=-\left.\frac{\partial L(y_i,F(x_i))}{\partial F(x_i)}\right|_{F=F_{m-1}}
$$

平方损失时它正比于普通残差 $y_i-F_{m-1}(x_i)$。

**第 3 步｜让新弱学习器拟合该方向并更新。** $F_m=F_{m-1}+\eta h_m$；递推展开后就是多个弱学习器的加法模型。

**把上面的关系收束为本章计算式：**

$$
F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta\,h_m(x)
$$

**符号解释：** $h_m$ 是第 $m$ 个弱学习器，$\eta$ 是学习率。

**代码对应：** 联合调节 `learning_rate` 与 `n_estimators`，用验证集观察过拟合。

**使用边界：** 较小学习率通常需要更多树；加法结构仍可能学习到数据偏差。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=83
)
boost = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.06,
    max_leaf_nodes=15,
    l2_regularization=1,
    early_stopping=True,
    random_state=83,
).fit(X_train, y_train)
prob = boost.predict_proba(X_test)[:, 1]
print(
    "迭代轮数:",
    boost.n_iter_,
    " ROC-AUC:",
    round(roc_auc_score(y_test, prob), 3),
)


**练一练**：只把示例里的参数从「max_leaf_nodes=15」改成「max_leaf_nodes=31」，其余保持不变，看看 ROC-AUC 会怎么变。先猜猜：允许更多的叶子节点，模型会变得更灵活，还是在训练集上更容易过拟合？把「next_leaves」填好后运行下面单元格自检。


In [ ]:
# 请在下方填写代码
# 练一练：把示例里提升模型的 max_leaf_nodes 从 15 改成 31，观察 ROC-AUC 怎么变化。


In [ ]:
# 完整答案：把 max_leaf_nodes 从 15 改成 31，观察 ROC-AUC 是否有变化
_next_leaves = 31

_wide_boost = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.06,
    max_leaf_nodes=_next_leaves,
    l2_regularization=1,
    early_stopping=True,
    random_state=83,
).fit(X_train, y_train)
print(
    "31 叶 ROC-AUC:",
    round(roc_auc_score(y_test, _wide_boost.predict_proba(X_test)[:, 1]), 3),
)
print("15 叶 ROC-AUC:", round(roc_auc_score(y_test, prob), 3))


## 94.4 示例 2：学习率与迭代数

比较有限配置，正式调参应放进交叉验证。


In [ ]:
import pandas as pd

rows = []
for rate, iters in [(0.03, 300), (0.06, 150), (0.1, 100)]:
    m = HistGradientBoostingClassifier(
        learning_rate=rate, max_iter=iters, max_leaf_nodes=15, random_state=83
    ).fit(X_train, y_train)
    rows.append(
        [rate, iters, roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])]
    )
display(
    pd.DataFrame(rows, columns=["learning_rate", "max_iter", "ROC_AUC"]).round(
        3
    )
)


## 94.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 94.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 94.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 94.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 94.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 94.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 94.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 94.9 易错点提醒

- 在小测试集上细调大量参数
- 使用过深基学习器导致过拟合
- 把训练轮数当成模型树深
- 未与简单线性和森林基线比较


## 94.10 练习与作业

1. 比较 max_leaf_nodes 为 5、15、31
2. 固定学习率和迭代次数
3. 报告测试 ROC-AUC

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 94.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“比较 max_leaf_nodes 为 5、15、31”。
2. **独立完成**：不复制示例代码，完成“固定学习率和迭代次数”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“报告测试 ROC-AUC”，用一两句话说明你修改了什么。

### 94.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 94.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
practice_rows = []
for leaves in [5, 15, 31]:
    m = HistGradientBoostingClassifier(
        max_leaf_nodes=leaves,
        max_iter=150,
        learning_rate=0.06,
        random_state=83,
    ).fit(X_train, y_train)
    practice_rows.append(
        [leaves, roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])]
    )
practice_result = pd.DataFrame(practice_rows, columns=["leaves", "auc"])
display(practice_result.round(3))


## 94.12 小结

学习梯度提升逐轮拟合残差的思想，使用 HistGradientBoosting 处理表格数据，并通过学习率与迭代次数控制复杂度。


### 94.12.1 你已经掌握

- 理解 boosting 的串行加法模型
- 训练 HistGradientBoostingClassifier
- 解释 learning_rate 和 max_iter
- 使用早停与验证集控制过拟合


### 94.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 94.12.3 需要注意

- 在小测试集上细调大量参数
- 使用过深基学习器导致过拟合
- 把训练轮数当成模型树深
- 未与简单线性和森林基线比较


### 94.12.4 完成检查

- [ ] 能够理解 boosting 的串行加法模型
- [ ] 能够训练 HistGradientBoostingClassifier
- [ ] 能够解释 learning_rate 和 max_iter
- [ ] 能够使用早停与验证集控制过拟合


### 94.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
